# 04_New_Models_Comparison.ipynb

This notebook implements and evaluates two new alternative models to address the J-shaped distribution of star ratings:
1. **Ordinal Logistic Regression** (Proportional Odds Model)
2. **Binary Logistic Regression** (5-stars vs. 1-4 stars)

We use the same 5-fold cross-validation setup (random_state=42) and GLASSO-selected interactions as the existing linear models to ensure comparability.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import pickle
import ast

# Add parent directory to path for imports
sys.path.append(os.path.abspath('..'))

from src.modeling_alternative import run_ordinal_cv, run_binary_cv, get_existing_results
from src.modeling import prepare_raw_modeling_data

# 1. Load Local Data
data_path = '../data/Seminar_Amazon_Results_FULL.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"✅ Local data loaded: {len(df)} rows.")
    
    # Parse stringified lists of tuples if they are strings
    if isinstance(df['aspect_sentiments'].iloc[0], str):
        print("Parsing aspect_sentiments column...")
        df['aspect_sentiments'] = df['aspect_sentiments'].apply(ast.literal_eval)
else:
    print(f"❌ Error: File not found at {data_path}. Please ensure the CSV is in the './data/' folder.")

c:\Users\Nativ\Documents\seminar\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Local data loaded: 701528 rows.
Parsing aspect_sentiments column...


## Part 1: Ordinal Logistic Regression
We treat star ratings as ordered categories.

In [2]:
print("Running Ordinal Logistic Regression CV...")
ordinal_results = run_ordinal_cv(df)

# Save results
with open('../model_ordinal_results.pkl', 'wb') as f:
    pickle.dump(ordinal_results, f)

def summarize_ordinal(res_list):
    df_res = pd.DataFrame(res_list)
    return df_res.mean()

ord_add_summary = summarize_ordinal(ordinal_results['additive'])
ord_int_summary = summarize_ordinal(ordinal_results['interaction'])

print("\nOrdinal Additive Summary:")
print(ord_add_summary)
print("\nOrdinal Interaction Summary:")
print(ord_int_summary)

2026-05-31 12:30:23,735 - INFO - Pivoting raw sentiment features...


Running Ordinal Logistic Regression CV...


2026-05-31 12:30:24,849 - INFO - Sparsity Filter: Dropped 379395 empty reviews. Remaining valid reviews: 322133
2026-05-31 12:30:24,851 - INFO - Starting Ordinal Logistic Regression Cross-Validation...
2026-05-31 12:30:24,876 - INFO - Processing Fold 1/5...
2026-05-31 12:32:25,527 - INFO - Standardizing feature matrix for 7 active aspects...
2026-05-31 12:32:25,556 - INFO - Estimating precision matrix using GraphicalLasso...
2026-05-31 12:32:26,378 - INFO - Generating interaction features from 12 network edges...
2026-05-31 12:32:26,394 - INFO - Generating interaction features from 12 network edges...
2026-05-31 12:40:03,278 - INFO - Processing Fold 2/5...
2026-05-31 12:41:35,381 - INFO - Standardizing feature matrix for 7 active aspects...
2026-05-31 12:41:35,415 - INFO - Estimating precision matrix using GraphicalLasso...
2026-05-31 12:41:36,259 - INFO - Generating interaction features from 12 network edges...
2026-05-31 12:41:36,274 - INFO - Generating interaction features from 12 n


Ordinal Additive Summary:
log_likelihood   -270355.407622
aic               540732.815243
rps                    0.131469
dtype: float64

Ordinal Interaction Summary:
log_likelihood   -270247.784490
aic               540540.368981
rps                    0.131445
dtype: float64


## Part 2: Binary Logistic Regression
We predict whether a review is a 5-star review (1) or not (0).

In [3]:
print("Running Binary Logistic Regression CV...")
binary_results = run_binary_cv(df)

# Save results
with open('../model_binary_results.pkl', 'wb') as f:
    pickle.dump(binary_results, f)

def summarize_binary(res_list):
    df_res = pd.DataFrame(res_list)
    return df_res.mean()

bin_add_summary = summarize_binary(binary_results['additive'])
bin_int_summary = summarize_binary(binary_results['interaction'])

print("\nBinary Additive Summary:")
print(bin_add_summary)
print("\nBinary Interaction Summary:")
print(bin_int_summary)

2026-05-31 13:11:54,793 - INFO - Pivoting raw sentiment features...


Running Binary Logistic Regression CV...


2026-05-31 13:11:55,820 - INFO - Sparsity Filter: Dropped 379395 empty reviews. Remaining valid reviews: 322133
2026-05-31 13:11:55,820 - INFO - Starting Binary Logistic Regression Cross-Validation...
2026-05-31 13:11:55,836 - INFO - Processing Fold 1/5...
c:\Users\Nativ\Documents\seminar\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
2026-05-31 13:11:56,225 - INFO - Standardizing feature matrix for 7 active aspects...
2026-05-31 13:11:56,259 - INFO - Estimating precision matrix using GraphicalLasso...
2026-05-31 13:11:56,964 - INFO - Generating interaction features from 12 network edges...
2026-05-31 13:11:56,976 - INFO - Generating interaction features from 12


Binary Additive Summary:
accuracy    0.747191
roc_auc     0.795362
f1          0.794392
dtype: float64

Binary Interaction Summary:
accuracy    0.745770
roc_auc     0.794359
f1          0.794064
dtype: float64


## Part 3: Final Comparison Table
Consolidating all models.

In [4]:
existing = get_existing_results()

summary_data = []

# Linear Models
summary_data.append({
    'Model': 'Linear Additive',
    'Type': 'Continuous',
    'Metric 1 (RMSE/Acc)': existing['Linear Additive']['Avg RMSE'],
    'Metric 2 (AdjR2/AUC)': existing['Linear Additive']['Avg Adj R2'],
    'Complexity (BIC/AIC)': existing['Linear Additive']['Full BIC']
})
summary_data.append({
    'Model': 'Linear Interaction',
    'Type': 'Continuous',
    'Metric 1 (RMSE/Acc)': existing['Linear Interaction']['Avg RMSE'],
    'Metric 2 (AdjR2/AUC)': existing['Linear Interaction']['Avg Adj R2'],
    'Complexity (BIC/AIC)': existing['Linear Interaction']['Full BIC']
})

# Ordinal Models
summary_data.append({
    'Model': 'Ordinal Additive',
    'Type': 'Categorical',
    'Metric 1 (RMSE/Acc)': ord_add_summary['rps'], # Using RPS for Metric 1
    'Metric 2 (AdjR2/AUC)': ord_add_summary['log_likelihood'],
    'Complexity (BIC/AIC)': ord_add_summary['aic']
})
summary_data.append({
    'Model': 'Ordinal Interaction',
    'Type': 'Categorical',
    'Metric 1 (RMSE/Acc)': ord_int_summary['rps'],
    'Metric 2 (AdjR2/AUC)': ord_int_summary['log_likelihood'],
    'Complexity (BIC/AIC)': ord_int_summary['aic']
})

# Binary Models
summary_data.append({
    'Model': 'Binary Additive',
    'Type': 'Binary',
    'Metric 1 (RMSE/Acc)': bin_add_summary['accuracy'],
    'Metric 2 (AdjR2/AUC)': bin_add_summary['roc_auc'],
    'Complexity (BIC/AIC)': bin_add_summary['f1']
})
summary_data.append({
    'Model': 'Binary Interaction',
    'Type': 'Binary',
    'Metric 1 (RMSE/Acc)': bin_int_summary['accuracy'],
    'Metric 2 (AdjR2/AUC)': bin_int_summary['roc_auc'],
    'Complexity (BIC/AIC)': bin_int_summary['f1']
})

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*90)
print("FINAL MODEL COMPARISON SUMMARY")
print("="*90)
print(summary_df.to_string(index=False))
print("="*90)

print("\nNote: Metrics vary by model type. Linear uses RMSE/AdjR2/BIC. Ordinal uses RPS/LL/AIC. Binary uses Acc/AUC/F1.")


FINAL MODEL COMPARISON SUMMARY
              Model        Type  Metric 1 (RMSE/Acc)  Metric 2 (AdjR2/AUC)  Complexity (BIC/AIC)
    Linear Additive  Continuous             1.214400              0.335100          1.039391e+06
 Linear Interaction  Continuous             1.203400              0.347000          1.032536e+06
   Ordinal Additive Categorical             0.131469        -270355.407622          5.407328e+05
Ordinal Interaction Categorical             0.131445        -270247.784490          5.405404e+05
    Binary Additive      Binary             0.747191              0.795362          7.943923e-01
 Binary Interaction      Binary             0.745770              0.794359          7.940635e-01

Note: Metrics vary by model type. Linear uses RMSE/AdjR2/BIC. Ordinal uses RPS/LL/AIC. Binary uses Acc/AUC/F1.
